# Food Delivery ETA Prediction - Feature Engineering

**Notebook:** 04_feature_engineering.ipynb
**Purpose:** Create meaningful features that may improve food delivery ETA prediction

This notebook creates domain-relevant features based on the cleaned dataset to improve model performance.

## Feature Engineering Objective

Create meaningful features that capture domain knowledge and relationships relevant to food delivery time prediction:
- Interaction features between related variables
- Ratio/derived features where meaningful
- Time-related features if applicable
- Domain-specific features based on delivery logistics

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


## Load Clean Dataset

In [2]:
df = pd.read_csv('../data/processed/food_delivery_clean.csv')
print(f"Clean dataset loaded: {df.shape}")

Clean dataset loaded: (1000, 9)


## Baseline Feature Set

In [3]:
print("Baseline Features:")
print(df.columns.tolist())
print(f"\nTotal baseline features: {df.shape[1]}")

Baseline Features:
['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time_min']

Total baseline features: 9


## Domain Understanding

Food delivery time depends on:
- Distance to customer (strong correlation: 0.78)
- Preparation time at restaurant (moderate correlation: 0.31)
- Courier experience (weak correlation: -0.09)
- Weather conditions (show variation in delivery times)
- Traffic levels (distinct delivery time patterns)
- Time of day (affects delivery duration)
- Vehicle type (some impact on delivery time)

Potential relationships to explore:
- Distance × Traffic interaction
- Distance × Weather interaction
- Preparation time × Distance
- Courier experience × Distance
- Rush hour indicator from Time_of_Day

## Candidate Feature Engineering

In [4]:
df_engineered = df.copy()
print("Starting feature engineering...")

Starting feature engineering...


## Interaction Features

In [5]:
# Distance × Traffic interaction
# Encode traffic levels numerically for interaction
traffic_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
df_engineered['Traffic_Num'] = df_engineered['Traffic_Level'].map(traffic_mapping)
df_engineered['Distance_Traffic_Interaction'] = df_engineered['Distance_km'] * df_engineered['Traffic_Num']

# Distance × Preparation Time interaction
df_engineered['Distance_Preparation_Interaction'] = df_engineered['Distance_km'] * df_engineered['Preparation_Time_min']

# Courier Experience × Distance interaction
df_engineered['Experience_Distance_Interaction'] = df_engineered['Courier_Experience_yrs'] * df_engineered['Distance_km']

print("Interaction features created:")
print("- Distance_Traffic_Interaction: Distance × Traffic Level")
print("- Distance_Preparation_Interaction: Distance × Preparation Time")
print("- Experience_Distance_Interaction: Experience × Distance")

print("\nWHY these interactions?")
print("- Distance × Traffic: Longer distances in high traffic may have disproportionate impact")
print("- Distance × Prep Time: Combined effect of distance and restaurant preparation")
print("- Experience × Distance: Experienced couriers may handle long distances better")

Interaction features created:
- Distance_Traffic_Interaction: Distance × Traffic Level
- Distance_Preparation_Interaction: Distance × Preparation Time
- Experience_Distance_Interaction: Experience × Distance

WHY these interactions?
- Distance × Traffic: Longer distances in high traffic may have disproportionate impact
- Distance × Prep Time: Combined effect of distance and restaurant preparation
- Experience × Distance: Experienced couriers may handle long distances better


## Ratio/Derived Features Where Meaningful

In [6]:
# Distance per unit of preparation time (delivery speed metric)
df_engineered['Distance_per_Preparation'] = df_engineered['Distance_km'] / (df_engineered['Preparation_Time_min'] + 1)

# Experience per distance (courier efficiency metric)
df_engineered['Experience_per_Distance'] = df_engineered['Courier_Experience_yrs'] / (df_engineered['Distance_km'] + 1)

# Total estimated time (preparation + typical delivery based on distance)
# Assuming average speed of 20 km/h for delivery
df_engineered['Estimated_Delivery_Time'] = df_engineered['Distance_km'] / 20 * 60  # in minutes
df_engineered['Total_Estimated_Time'] = df_engineered['Preparation_Time_min'] + df_engineered['Estimated_Delivery_Time']

print("Ratio/Derived features created:")
print("- Distance_per_Preparation: Distance relative to preparation time")
print("- Experience_per_Distance: Courier efficiency metric")
print("- Estimated_Delivery_Time: Time based on distance alone")
print("- Total_Estimated_Time: Preparation + estimated delivery")

print("\nWHY these ratios?")
print("- Distance_per_Preparation: Captures restaurant efficiency")
print("- Experience_per_Distance: Captures courier efficiency")
print("- Total_Estimated_Time: Baseline expectation for comparison")

Ratio/Derived features created:
- Distance_per_Preparation: Distance relative to preparation time
- Experience_per_Distance: Courier efficiency metric
- Estimated_Delivery_Time: Time based on distance alone
- Total_Estimated_Time: Preparation + estimated delivery

WHY these ratios?
- Distance_per_Preparation: Captures restaurant efficiency
- Experience_per_Distance: Captures courier efficiency
- Total_Estimated_Time: Baseline expectation for comparison


## Time-related Features Where Applicable

In [7]:
# Encode time of day numerically
time_mapping = {'Morning': 1, 'Afternoon': 2, 'Evening': 3, 'Night': 4}
df_engineered['Time_of_Day_Num'] = df_engineered['Time_of_Day'].map(time_mapping)

# Create rush hour indicator (assuming Evening is rush hour based on EDA)
df_engineered['Is_Rush_Hour'] = (df_engineered['Time_of_Day'] == 'Evening').astype(int)

# Create adverse weather indicator (Rainy, Snowy, Foggy, Windy)
adverse_weather = ['Rainy', 'Snowy', 'Foggy', 'Windy']
df_engineered['Is_Adverse_Weather'] = df_engineered['Weather'].isin(adverse_weather).astype(int)

print("Time-related features created:")
print("- Time_of_Day_Num: Numerical encoding of time periods")
print("- Is_Rush_Hour: Indicator for evening rush hour")
print("- Is_Adverse_Weather: Indicator for adverse weather conditions")

print("\nWHY these features?")
print("- Time_of_Day_Num: Enables mathematical operations on time")
print("- Is_Rush_Hour: Captures peak delivery periods")
print("- Is_Adverse_Weather: Captures challenging delivery conditions")

Time-related features created:
- Time_of_Day_Num: Numerical encoding of time periods
- Is_Rush_Hour: Indicator for evening rush hour
- Is_Adverse_Weather: Indicator for adverse weather conditions

WHY these features?
- Time_of_Day_Num: Enables mathematical operations on time
- Is_Rush_Hour: Captures peak delivery periods
- Is_Adverse_Weather: Captures challenging delivery conditions


## Feature Distribution After Engineering

In [8]:
engineered_numerical = ['Distance_Traffic_Interaction', 'Distance_Preparation_Interaction', 
                        'Experience_Distance_Interaction', 'Distance_per_Preparation',
                        'Experience_per_Distance', 'Estimated_Delivery_Time', 'Total_Estimated_Time']

print("Engineered numerical features statistics:")
print(df_engineered[engineered_numerical].describe())

Engineered numerical features statistics:
       Distance_Traffic_Interaction  Distance_Preparation_Interaction  \
count                    970.000000                       1000.000000   
mean                      18.184021                        170.467900   
std                       13.676409                        128.683155   
min                        0.590000                          4.680000   
25%                        7.572500                         70.417500   
50%                       15.050000                        136.055000   
75%                       25.700000                        243.492500   
max                       59.970000                        558.830000   

       Experience_Distance_Interaction  Distance_per_Preparation  \
count                       970.000000               1000.000000   
mean                         45.906165                  0.691780   
std                          42.404152                  0.574560   
min                         

## Compare Original vs Engineered Features

In [9]:
print("Feature Comparison:")
print(f"\nOriginal features: {df.shape[1]}")
print(f"Engineered features: {df_engineered.shape[1]}")
print(f"New features added: {df_engineered.shape[1] - df.shape[1]}")

print("\nNew features:")
new_features = [col for col in df_engineered.columns if col not in df.columns]
for i, col in enumerate(new_features, 1):
    print(f"{i}. {col}")

Feature Comparison:

Original features: 9
Engineered features: 20
New features added: 11

New features:
1. Traffic_Num
2. Distance_Traffic_Interaction
3. Distance_Preparation_Interaction
4. Experience_Distance_Interaction
5. Distance_per_Preparation
6. Experience_per_Distance
7. Estimated_Delivery_Time
8. Total_Estimated_Time
9. Time_of_Day_Num
10. Is_Rush_Hour
11. Is_Adverse_Weather


## Feature Engineering Validation

In [10]:
# Check correlation of engineered features with target
target = 'Delivery_Time_min'
correlations = df_engineered[engineered_numerical + [target]].corr()[target].sort_values(ascending=False)

print("Correlation of engineered features with target:")
print(correlations)

# Check for infinite or NaN values
print("\nValidation checks:")
for col in engineered_numerical:
    inf_count = np.isinf(df_engineered[col]).sum()
    nan_count = df_engineered[col].isnull().sum()
    print(f"{col}: {inf_count} infinite values, {nan_count} NaN values")

Correlation of engineered features with target:
Delivery_Time_min                   1.000000
Total_Estimated_Time                0.841784
Estimated_Delivery_Time             0.780998
Distance_Preparation_Interaction    0.768055
Distance_Traffic_Interaction        0.698567
Experience_Distance_Interaction     0.414682
Distance_per_Preparation            0.351544
Experience_per_Distance            -0.493433
Name: Delivery_Time_min, dtype: float64

Validation checks:
Distance_Traffic_Interaction: 0 infinite values, 30 NaN values
Distance_Preparation_Interaction: 0 infinite values, 0 NaN values
Experience_Distance_Interaction: 0 infinite values, 30 NaN values
Distance_per_Preparation: 0 infinite values, 0 NaN values
Experience_per_Distance: 0 infinite values, 30 NaN values
Estimated_Delivery_Time: 0 infinite values, 0 NaN values
Total_Estimated_Time: 0 infinite values, 0 NaN values


## Final Candidate Features

In [11]:
print("Final Candidate Features:")
print(df_engineered.columns.tolist())
print(f"\nTotal features: {df_engineered.shape[1]}")

Final Candidate Features:
['Order_ID', 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time_min', 'Traffic_Num', 'Distance_Traffic_Interaction', 'Distance_Preparation_Interaction', 'Experience_Distance_Interaction', 'Distance_per_Preparation', 'Experience_per_Distance', 'Estimated_Delivery_Time', 'Total_Estimated_Time', 'Time_of_Day_Num', 'Is_Rush_Hour', 'Is_Adverse_Weather']

Total features: 20


## Save Engineered Dataset

In [12]:
# Save engineered dataset
df_engineered.to_csv('../data/processed/food_delivery_features.csv', index=False)
print("Engineered dataset saved to: ../data/processed/food_delivery_features.csv")

Engineered dataset saved to: ../data/processed/food_delivery_features.csv


## Create Reusable Feature Engineering Function

In [13]:
def engineer_features(df):
    """
    Engineer features for food delivery ETA prediction.
    
    Args:
        df: Clean dataframe
    
    Returns:
        Dataframe with engineered features
    """
    df_engineered = df.copy()
    
    # Interaction features
    traffic_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
    df_engineered['Traffic_Num'] = df_engineered['Traffic_Level'].map(traffic_mapping)
    df_engineered['Distance_Traffic_Interaction'] = df_engineered['Distance_km'] * df_engineered['Traffic_Num']
    df_engineered['Distance_Preparation_Interaction'] = df_engineered['Distance_km'] * df_engineered['Preparation_Time_min']
    df_engineered['Experience_Distance_Interaction'] = df_engineered['Courier_Experience_yrs'] * df_engineered['Distance_km']
    
    # Ratio features
    df_engineered['Distance_per_Preparation'] = df_engineered['Distance_km'] / (df_engineered['Preparation_Time_min'] + 1)
    df_engineered['Experience_per_Distance'] = df_engineered['Courier_Experience_yrs'] / (df_engineered['Distance_km'] + 1)
    
    # Estimated time features
    df_engineered['Estimated_Delivery_Time'] = df_engineered['Distance_km'] / 20 * 60
    df_engineered['Total_Estimated_Time'] = df_engineered['Preparation_Time_min'] + df_engineered['Estimated_Delivery_Time']
    
    # Time-related features
    time_mapping = {'Morning': 1, 'Afternoon': 2, 'Evening': 3, 'Night': 4}
    df_engineered['Time_of_Day_Num'] = df_engineered['Time_of_Day'].map(time_mapping)
    df_engineered['Is_Rush_Hour'] = (df_engineered['Time_of_Day'] == 'Evening').astype(int)
    
    # Weather indicator
    adverse_weather = ['Rainy', 'Snowy', 'Foggy', 'Windy']
    df_engineered['Is_Adverse_Weather'] = df_engineered['Weather'].isin(adverse_weather).astype(int)
    
    return df_engineered

print("Reusable feature engineering function created.")

Reusable feature engineering function created.


## Feature Engineering Conclusions

In [14]:
print("="*60)
print("FEATURE ENGINEERING CONCLUSIONS")
print("="*60)

print("\nENGINEERED FEATURES:")
print("- Distance_Traffic_Interaction: Captures distance impact under traffic conditions")
print("- Distance_Preparation_Interaction: Combined distance and preparation effect")
print("- Experience_Distance_Interaction: Courier experience relative to distance")
print("- Distance_per_Preparation: Restaurant efficiency metric")
print("- Experience_per_Distance: Courier efficiency metric")
print("- Estimated_Delivery_Time: Baseline delivery time from distance")
print("- Total_Estimated_Time: Complete baseline estimate")
print("- Time_of_Day_Num: Numerical time encoding")
print("- Is_Rush_Hour: Peak period indicator")
print("- Is_Adverse_Weather: Challenging conditions indicator")

print("\nFEATURE SELECTION INSIGHTS:")
print("- All engineered features have logical domain meaning")
print("- Interaction features capture nonlinear relationships")
print("- Ratio features capture efficiency metrics")
print("- Time features capture temporal patterns")
print("- No data leakage introduced")
print("- Features ready for selection process")

print("\n" + "="*60)
print("Feature engineering complete. Ready for feature selection.")
print("="*60)

FEATURE ENGINEERING CONCLUSIONS

ENGINEERED FEATURES:
- Distance_Traffic_Interaction: Captures distance impact under traffic conditions
- Distance_Preparation_Interaction: Combined distance and preparation effect
- Experience_Distance_Interaction: Courier experience relative to distance
- Distance_per_Preparation: Restaurant efficiency metric
- Experience_per_Distance: Courier efficiency metric
- Estimated_Delivery_Time: Baseline delivery time from distance
- Total_Estimated_Time: Complete baseline estimate
- Time_of_Day_Num: Numerical time encoding
- Is_Rush_Hour: Peak period indicator
- Is_Adverse_Weather: Challenging conditions indicator

FEATURE SELECTION INSIGHTS:
- All engineered features have logical domain meaning
- Interaction features capture nonlinear relationships
- Ratio features capture efficiency metrics
- Time features capture temporal patterns
- No data leakage introduced
- Features ready for selection process

Feature engineering complete. Ready for feature selection.